# C1 — Barely Viable Portfolio: Single Day Diagnostic

**Purpose:** Verify that the 100 kW / 200 kWh portfolio (C1) can only reserve FCR-D in one direction per hour due to the NEM cross-direction constraint.

**Config:** 10 ECs × 10 kW / 20 kWh, p_min = 100 kW (= full inverter capacity)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, time
from datetime import date as dt_date

import pyomo.environ as pyo

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.max_open_warning': 0, 'figure.dpi': 120})

print('All imports OK')

## 1. Configuration & Parameters (Table 2)

In [ ]:
# ── Paths (adjust to your layout) ─────────────────────────────────────────
BASE_DIR       = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_OUT_DIR   = os.path.join(BASE_DIR, 'data', 'data_out')
FIGURES_DIR    = os.path.join(BASE_DIR, 'figures')
COMBINED_CSV   = os.path.join(DATA_OUT_DIR, 'combined_2025_with_frequency.csv')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Portfolio parameters (Table 2) ────────────────────────────────────────
N_EC           = 10        # number of energy communities
MAX_POWER_EC   = 10         # C1: 10 kW per EC       # kW per EC battery
CAPACITY_EC    = 20         # C1: 20 kWh per EC       # kWh per EC battery
ETA            = 0.95      # one-way efficiency; round-trip = eta^2
SOC_INIT_FRAC  = 0.50      # initial & terminal SOC fraction
P_MIN_BID      = 100       # kW minimum FCR-D bid in DK2
T_SUSTAIN      = 1/3       # hours sustain duration (regulatory min, Energinet 2023)
UPSILON_P      = 0.20      # LER NEM cross-direction power reservation factor (Energinet, 2023)
T_MAX          = 24        # hours per day

# Derived
P_MAX_PORTFOLIO = N_EC * MAX_POWER_EC   # 1000 kW
S_MAX_PORTFOLIO = N_EC * CAPACITY_EC     # 2000 kWh
SOC_INIT_KWH    = SOC_INIT_FRAC * CAPACITY_EC  # 100 kWh per EC

# ── Synthetic portfolio (deterministic seed) ──────────────────────────────
RNG_SEED       = 42
PROB_B_TYPE    = 0.60
SCALE_MU       = 1.0
SCALE_SIGMA    = 1.0

ORE_TO_DKK     = 1.0 / 100.0   # prices in CSV are øre/kWh → DKK/kWh

print(f'Portfolio: {N_EC} ECs × {MAX_POWER_EC} kW / {CAPACITY_EC} kWh = '
      f'{P_MAX_PORTFOLIO} kW / {S_MAX_PORTFOLIO} kWh')


## 2. Load & prepare data

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────
df_raw = pd.read_csv(COMBINED_CSV, parse_dates=['hour_utc'])
df_raw = df_raw.sort_values(['hour_utc', 'ec_id']).reset_index(drop=True)
print(f'Loaded {len(df_raw)} rows  |  ec_ids: {sorted(df_raw["ec_id"].unique())}  |  '
      f'{df_raw["hour_utc"].min()} → {df_raw["hour_utc"].max()}')

# ── Separate market data (one row per hour) from EC profiles ──────────────
# Market columns are the same for both ec_id rows in each hour;
# take from the first row per hour.
df_market = df_raw.groupby('hour_utc').first().reset_index()

# Convert prices from øre/kWh → DKK/kWh
price_cols_ore = [
    'buy_price_inkl_vat_ore_kwh',
    'sell_price_inkl_vat_ore_kwh',
    'price_ore_kwh_fcr_d_upp__d_1_early',
    'price_ore_kwh_fcr_d_upp__d_1_late',
    'price_ore_kwh_fcr_d_ned__d_1_early',
    'price_ore_kwh_fcr_d_ned__d_1_late',
]
for col in price_cols_ore:
    df_market[col.replace('ore', 'dkk')] = df_market[col] * ORE_TO_DKK

# Rename for convenience
df_market.rename(columns={
    'buy_price_inkl_vat_ore_kwh':           'buy_ore',
    'sell_price_inkl_vat_ore_kwh':          'sell_ore',
}, inplace=True)

# DKK price columns (created above)
COL_BUY       = 'buy_price_inkl_vat_dkk_kwh'
COL_SELL      = 'sell_price_inkl_vat_dkk_kwh'
COL_UP_EARLY  = 'price_dkk_kwh_fcr_d_upp__d_1_early'
COL_UP_LATE   = 'price_dkk_kwh_fcr_d_upp__d_1_late'
COL_DOWN_EARLY= 'price_dkk_kwh_fcr_d_ned__d_1_early'
COL_DOWN_LATE = 'price_dkk_kwh_fcr_d_ned__d_1_late'
COL_ACT_UP    = 'y_act_fcrd_up'
COL_ACT_DOWN  = 'y_act_fcrd_down'

# Buyback price = max(early, late) per direction
df_market['buyback_up']   = df_market[[COL_UP_EARLY,   COL_UP_LATE]].max(axis=1)
df_market['buyback_down'] = df_market[[COL_DOWN_EARLY, COL_DOWN_LATE]].max(axis=1)

# Acceptance flag: accepted where price > 0 (Assumption A1)
df_market['y_acc_up']   = (df_market[COL_UP_EARLY]   > 0).astype(float)
df_market['y_acc_down'] = (df_market[COL_DOWN_EARLY] > 0).astype(float)

df_market['date'] = df_market['hour_utc'].dt.date

print(f'Market data: {len(df_market)} hours')
print(f'Sample buy price (DKK/kWh): {df_market[COL_BUY].iloc[0]:.4f}')
print(f'Sample FCR-D Up early (DKK/kWh): {df_market[COL_UP_EARLY].iloc[0]:.4f}')

In [ ]:
# ── Extract base EC profiles (b-type and s-type) per hour ────────────────
# The CSV has one row per (hour, ec_id). Pivot to get daily profile shapes.
df_b = df_raw[df_raw['ec_id'] == 'b'][['hour_utc', 'consumption', 'pv_production_kwh']].copy()
df_b = df_b.rename(columns={'consumption': 'load_b', 'pv_production_kwh': 'pv_b'})

df_s = df_raw[df_raw['ec_id'] == 's'][['hour_utc', 'consumption', 'pv_production_kwh']].copy()
df_s = df_s.rename(columns={'consumption': 'load_s', 'pv_production_kwh': 'pv_s'})

# Merge onto market data
df_all = df_market.merge(df_b, on='hour_utc', how='left')
df_all = df_all.merge(df_s, on='hour_utc', how='left')

# Fill any NaN profiles with 0
for c in ['load_b', 'pv_b', 'load_s', 'pv_s']:
    df_all[c] = df_all[c].fillna(0.0)

print(f'Combined data: {len(df_all)} rows')
print(df_all[['hour_utc', 'load_b', 'pv_b', 'load_s', 'pv_s',
              COL_BUY, COL_UP_EARLY]].head(5))

## 3. Portfolio construction (deterministic)

In [ ]:
def generate_portfolio(n_ec, prob_b, scale_mu, scale_sigma, seed):
    """
    Deterministic synthetic portfolio.
    Each EC is assigned b-type or s-type, with a positive scale factor.
    """
    rng = np.random.default_rng(seed)
    ec_types = []
    scale_factors = []
    for _ in range(n_ec):
        ec_types.append('b' if rng.random() < prob_b else 's')
        while True:
            s = rng.normal(scale_mu, scale_sigma)
            if s > 0:
                scale_factors.append(s)
                break
    return ec_types, np.array(scale_factors)


ec_types, scale_factors = generate_portfolio(
    N_EC, PROB_B_TYPE, SCALE_MU, SCALE_SIGMA, RNG_SEED
)

n_b = sum(t == 'b' for t in ec_types)
n_s = N_EC - n_b
print(f'Portfolio: {n_b} b-type, {n_s} s-type')
print(f'Scale factors: {np.round(scale_factors, 3)}')

In [ ]:
def get_ec_profiles(df_day, ec_types, scale_factors):
    """
    Build (N_EC, 24) arrays for load and PV from the day's base profiles.
    Each EC gets its type's base profile × its scale factor.
    """
    n_ec = len(ec_types)
    loads = np.zeros((n_ec, 24))
    pvs   = np.zeros((n_ec, 24))
    
    load_b = df_day['load_b'].values
    pv_b   = df_day['pv_b'].values
    load_s = df_day['load_s'].values
    pv_s   = df_day['pv_s'].values
    
    for e in range(n_ec):
        if ec_types[e] == 'b':
            loads[e] = load_b * scale_factors[e]
            pvs[e]   = pv_b   * scale_factors[e]
        else:
            loads[e] = load_s * scale_factors[e]
            pvs[e]   = pv_s   * scale_factors[e]
    
    return loads, pvs

print('get_ec_profiles() defined')

## 4. MILP Model: Early Auction (§2.1)

Perfect forecasts: actual values used as-is.

In [ ]:
def build_early_model(loads, pvs, buy_price, sell_price,
                      fcrd_up_price, fcrd_down_price,
                      act_frac_up, act_frac_down,
                      y_acc_up, y_acc_down):
    """
    Build the early auction MILP (§2.1).
    loads, pvs: (N_EC, 24).  All price/frac arrays: (24,).
    All values are PERFECT (actual) — no forecast error.
    """
    n_ec  = loads.shape[0]
    b_max = MAX_POWER_EC
    s_max = CAPACITY_EC
    eta   = ETA
    soc0  = SOC_INIT_KWH
    p_min = P_MIN_BID
    p_max = P_MAX_PORTFOLIO
    t_sus = T_SUSTAIN
    E = range(n_ec)
    T = range(T_MAX)

    m = pyo.ConcreteModel('Early')
    m.E = pyo.Set(initialize=E)
    m.T = pyo.Set(initialize=T)

    # Decision variables (all >= 0 unless stated)
    m.p_im       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_ex       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_ch       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_dis      = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.SOC        = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_net      = pyo.Var(m.E, m.T, within=pyo.Reals)
    m.z_up       = pyo.Var(m.T, within=pyo.Binary)
    m.z_down     = pyo.Var(m.T, within=pyo.Binary)

    # ── Objective (m2a) ───────────────────────────────────────────────────
    def obj_rule(m):
        return sum(
            buy_price[t]  * (sum(m.p_im[e,t] for e in E) - sum(m.p_down_act[e,t] for e in E))
          - sell_price[t] * (sum(m.p_ex[e,t] for e in E) - sum(m.p_up_act[e,t]   for e in E))
          - fcrd_up_price[t]   * sum(m.p_up_res[e,t]   for e in E)
          - fcrd_down_price[t] * sum(m.p_down_res[e,t] for e in E)
          for t in T)
    m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # (m1b) p_net = p_im - p_ex
    m.c_net = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] - m.p_im[e,t] + m.p_ex[e,t] == 0)

    # (c2i) Power balance:  p_net + PV - D - (b_ch + p_down_act) + (b_dis + p_up_act) = 0
    m.c_pb = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] + pvs[e,t] - loads[e,t]
            - (m.b_ch[e,t] + m.p_down_act[e,t])
            + (m.b_dis[e,t] + m.p_up_act[e,t]) == 0)

    # (c3bup/down) Activation = y_acc * y_act * reservation
    m.c_act_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_act[e,t] == y_acc_up[t] * act_frac_up[t] * m.p_up_res[e,t])
    m.c_act_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_act[e,t] == y_acc_down[t] * act_frac_down[t] * m.p_down_res[e,t])

    # (m2k) SOC at t=0
    m.c_soc0 = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e,0] == soc0
            + eta * (m.b_ch[e,0] + m.p_down_act[e,0])
            - (1.0/eta) * (m.b_dis[e,0] + m.p_up_act[e,0]))

    # (m2j) SOC dynamics for t in 1..T_MAX-2
    def soc_dyn(m, e, t):
        if t == 0:
            return pyo.Constraint.Skip
        return m.SOC[e,t] == m.SOC[e,t-1] \
            + eta * (m.b_ch[e,t] + m.p_down_act[e,t]) \
            - (1.0/eta) * (m.b_dis[e,t] + m.p_up_act[e,t])
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dyn)

    # (c1l) Terminal SOC = soc0
    m.c_soc_end = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e, T_MAX-1] == soc0)

    # (m13 up)  SOC >= y_acc * T_sus / eta * p_up_res
    m.c_sus_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] >= y_acc_up[t] * t_sus / eta * m.p_up_res[e,t])

    # (m13 down) S_max - SOC >= y_acc * T_sus * eta * p_down_res
    m.c_sus_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: s_max - m.SOC[e,t] >= y_acc_down[t] * t_sus * eta * m.p_down_res[e,t])

    # (m-ler-down) LER power: b_ch + p_down_res + 0.20*p_up_res <= b_max
    #   NEM cross-direction reservation per Energinet (2023), Thingvad et al. (2024) Eq. 11
    m.c_ch_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_ch[e,t] + m.p_down_res[e,t] + UPSILON_P * m.p_up_res[e,t] <= b_max)

    # (m-ler-up) LER power: b_dis + p_up_res + 0.20*p_down_res <= b_max
    #   NEM cross-direction reservation per Energinet (2023), Thingvad et al. (2024) Eq. 10
    m.c_dis_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_dis[e,t] + m.p_up_res[e,t] + UPSILON_P * m.p_down_res[e,t] <= b_max)

    # SOC upper bound
    m.c_soc_cap = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] <= s_max)

    # (m1up/m1down) Min bid size MILP
    m.c_bid_up_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min * m.z_up[t] <= sum(m.p_up_res[e,t] for e in E))
    m.c_bid_up_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_up_res[e,t] for e in E) <= p_max * m.z_up[t])
    m.c_bid_dn_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min * m.z_down[t] <= sum(m.p_down_res[e,t] for e in E))
    m.c_bid_dn_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_down_res[e,t] for e in E) <= p_max * m.z_down[t])

    return m

print('build_early_model() defined')


## 5. MILP Model: Late Auction (§2.2)

Takes accepted early bids as given. Can top-up or cancel (buy back) per hour.

In [ ]:
def build_late_model(loads, pvs, buy_price, sell_price,
                     fcrd_up_early_price, fcrd_down_early_price,
                     fcrd_up_late_price, fcrd_down_late_price,
                     buyback_up_price, buyback_down_price,
                     act_frac_up, act_frac_down,
                     y_acc_up, y_acc_down,
                     early_up_acc, early_down_acc):
    """
    Build the late auction MILP (§2.2).
    early_up_acc, early_down_acc: (N_EC, 24) accepted early bids.
    """
    n_ec  = loads.shape[0]
    b_max = MAX_POWER_EC
    s_max = CAPACITY_EC
    eta   = ETA
    soc0  = SOC_INIT_KWH
    p_min = P_MIN_BID
    p_max = P_MAX_PORTFOLIO
    t_sus = T_SUSTAIN
    E = range(n_ec)
    T = range(T_MAX)

    m = pyo.ConcreteModel('Late')
    m.E = pyo.Set(initialize=E)
    m.T = pyo.Set(initialize=T)

    # Variables
    m.p_im       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_ex       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_ch       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_dis      = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.SOC        = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_net      = pyo.Var(m.E, m.T, within=pyo.Reals)
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # final total
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.a_up       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # top-up
    m.a_down     = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.c_up       = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # cancel
    m.c_down     = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.w_up       = pyo.Var(m.T, within=pyo.Binary)  # 1 if topping up
    m.w_down     = pyo.Var(m.T, within=pyo.Binary)
    m.z_up       = pyo.Var(m.T, within=pyo.Binary)  # 1 if final total > 0
    m.z_down     = pyo.Var(m.T, within=pyo.Binary)

    # Link final = early_accepted + top-up - cancel
    m.c_link_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_res[e,t] == early_up_acc[e,t] + m.a_up[e,t] - m.c_up[e,t])
    m.c_link_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_res[e,t] == early_down_acc[e,t] + m.a_down[e,t] - m.c_down[e,t])

    # ── Objective (m2a late) ──────────────────────────────────────────────
    def obj_rule(m):
        return sum(
            buy_price[t]  * (sum(m.p_im[e,t] for e in E) - sum(m.p_down_act[e,t] for e in E))
          - sell_price[t] * (sum(m.p_ex[e,t] for e in E) - sum(m.p_up_act[e,t]   for e in E))
          - fcrd_up_early_price[t]   * sum(early_up_acc[e,t]  for e in E)
          - fcrd_down_early_price[t] * sum(early_down_acc[e,t] for e in E)
          + buyback_up_price[t]   * sum(m.c_up[e,t]   for e in E)
          + buyback_down_price[t] * sum(m.c_down[e,t] for e in E)
          - fcrd_up_late_price[t]   * sum(m.a_up[e,t]   for e in E)
          - fcrd_down_late_price[t] * sum(m.a_down[e,t] for e in E)
          for t in T)
    m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # ── Same base constraints ─────────────────────────────────────────────
    m.c_net = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] - m.p_im[e,t] + m.p_ex[e,t] == 0)
    m.c_pb = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_net[e,t] + pvs[e,t] - loads[e,t]
            - (m.b_ch[e,t] + m.p_down_act[e,t])
            + (m.b_dis[e,t] + m.p_up_act[e,t]) == 0)
    m.c_act_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_up_act[e,t] == y_acc_up[t]*act_frac_up[t]*m.p_up_res[e,t])
    m.c_act_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.p_down_act[e,t] == y_acc_down[t]*act_frac_down[t]*m.p_down_res[e,t])

    m.c_soc0 = pyo.Constraint(m.E,
        rule=lambda m,e: m.SOC[e,0] == soc0
            + eta*(m.b_ch[e,0]+m.p_down_act[e,0])
            - (1.0/eta)*(m.b_dis[e,0]+m.p_up_act[e,0]))
    def soc_dyn(m,e,t):
        if t==0: return pyo.Constraint.Skip
        return m.SOC[e,t] == m.SOC[e,t-1] \
            + eta*(m.b_ch[e,t]+m.p_down_act[e,t]) \
            - (1.0/eta)*(m.b_dis[e,t]+m.p_up_act[e,t])
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dyn)
    m.c_soc_end = pyo.Constraint(m.E, rule=lambda m,e: m.SOC[e,T_MAX-1]==soc0)

    m.c_sus_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] >= y_acc_up[t]*t_sus/eta*m.p_up_res[e,t])
    m.c_sus_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: s_max - m.SOC[e,t] >= y_acc_down[t]*t_sus*eta*m.p_down_res[e,t])

    # (m-ler-down) LER power with NEM cross-direction reservation
    m.c_ch_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_ch[e,t] + m.p_down_res[e,t] + UPSILON_P * m.p_up_res[e,t] <= b_max)
    # (m-ler-up) LER power with NEM cross-direction reservation
    m.c_dis_lim = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.b_dis[e,t] + m.p_up_res[e,t] + UPSILON_P * m.p_down_res[e,t] <= b_max)
    m.c_soc_cap = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.SOC[e,t] <= s_max)

    # Min bid for top-ups
    m.c_tu_up_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min*m.w_up[t] <= sum(m.a_up[e,t] for e in E))
    m.c_tu_up_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.a_up[e,t] for e in E) <= p_max*m.w_up[t])
    m.c_tu_dn_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min*m.w_down[t] <= sum(m.a_down[e,t] for e in E))
    m.c_tu_dn_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.a_down[e,t] for e in E) <= p_max*m.w_down[t])

    # Min bid for final total reservation
    m.c_fin_up_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min*m.z_up[t] <= sum(m.p_up_res[e,t] for e in E))
    m.c_fin_up_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_up_res[e,t] for e in E) <= p_max*m.z_up[t])
    m.c_fin_dn_lo = pyo.Constraint(m.T,
        rule=lambda m,t: p_min*m.z_down[t] <= sum(m.p_down_res[e,t] for e in E))
    m.c_fin_dn_hi = pyo.Constraint(m.T,
        rule=lambda m,t: sum(m.p_down_res[e,t] for e in E) <= p_max*m.z_down[t])

    # Cannot cancel and top-up simultaneously
    m.c_nosim_up = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.c_up[e,t] <= early_up_acc[e,t]*(1-m.w_up[t]))
    m.c_nosim_dn = pyo.Constraint(m.E, m.T,
        rule=lambda m,e,t: m.c_down[e,t] <= early_down_acc[e,t]*(1-m.w_down[t]))

    return m

print('build_late_model() defined')


## 6. Solver & helpers

In [ ]:
from pyomo.environ import SolverFactory

SOLVER = 'appsi_highs'

def solve_model(model):
    res = SolverFactory(SOLVER).solve(model, tee=False)
    tc = res.solver.termination_condition
    if tc not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        raise RuntimeError(f'Solver: {tc}')
    return res

def xvar2d(var, n, t=T_MAX):
    a = np.zeros((n, t))
    for e in range(n):
        for h in range(t):
            a[e,h] = pyo.value(var[e,h])
    return a

print('Solver ready')

## 7. Daily solve (Early → Late, perfect forecasts)

In [ ]:
def solve_day(date_val, df_day, loads, pvs, run_base=True):
    """
    Solve sequential auction for one day with PERFECT forecasts.
    df_day: one-row-per-hour market data for this day (24 rows).
    loads, pvs: (N_EC, 24).
    """
    n_ec = loads.shape[0]

    buy   = df_day[COL_BUY].values
    sell  = df_day[COL_SELL].values
    up_e  = df_day[COL_UP_EARLY].values
    dn_e  = df_day[COL_DOWN_EARLY].values
    up_l  = df_day[COL_UP_LATE].values
    dn_l  = df_day[COL_DOWN_LATE].values
    af_u  = df_day[COL_ACT_UP].values
    af_d  = df_day[COL_ACT_DOWN].values
    ya_u  = df_day['y_acc_up'].values
    ya_d  = df_day['y_acc_down'].values
    bb_u  = df_day['buyback_up'].values
    bb_d  = df_day['buyback_down'].values

    # ── Early auction ─────────────────────────────────────────────────────
    me = build_early_model(loads, pvs, buy, sell, up_e, dn_e, af_u, af_d, ya_u, ya_d)
    solve_model(me)

    early_up  = xvar2d(me.p_up_res,   n_ec)
    early_dn  = xvar2d(me.p_down_res, n_ec)
    # Assumption A1: all accepted
    early_up_acc = early_up.copy()
    early_dn_acc = early_dn.copy()

    # ── Late auction ──────────────────────────────────────────────────────
    ml = build_late_model(loads, pvs, buy, sell,
                          up_e, dn_e, up_l, dn_l, bb_u, bb_d,
                          af_u, af_d, ya_u, ya_d,
                          early_up_acc, early_dn_acc)
    solve_model(ml)

    # Extract final results
    fin_up  = xvar2d(ml.p_up_res,   n_ec)
    fin_dn  = xvar2d(ml.p_down_res, n_ec)
    fin_im  = xvar2d(ml.p_im,       n_ec)
    fin_ex  = xvar2d(ml.p_ex,       n_ec)
    fin_soc = xvar2d(ml.SOC,        n_ec)
    fin_bch = xvar2d(ml.b_ch,       n_ec)
    fin_bdis= xvar2d(ml.b_dis,      n_ec)
    fin_uact= xvar2d(ml.p_up_act,   n_ec)
    fin_dact= xvar2d(ml.p_down_act, n_ec)
    tu_up   = xvar2d(ml.a_up,       n_ec)
    tu_dn   = xvar2d(ml.a_down,     n_ec)
    cn_up   = xvar2d(ml.c_up,       n_ec)
    cn_dn   = xvar2d(ml.c_down,     n_ec)

    # Aggregate across ECs
    A = lambda x: x.sum(axis=0)

    # Revenue components
    import_cost = (buy * (A(fin_im) - A(fin_dact))).sum()
    export_rev  = (sell * (A(fin_ex) - A(fin_uact))).sum()

    fcrd_up_early_rev   = (up_e * A(early_up_acc)).sum()
    fcrd_dn_early_rev   = (dn_e * A(early_dn_acc)).sum()
    fcrd_up_late_rev    = (up_l * A(tu_up)).sum()
    fcrd_dn_late_rev    = (dn_l * A(tu_dn)).sum()
    buyback_up_cost     = (bb_u * A(cn_up)).sum()
    buyback_dn_cost     = (bb_d * A(cn_dn)).sum()

    r = {
        'date': date_val,
        'net_energy_cost': import_cost - export_rev,
        'import_cost': import_cost,
        'export_rev': export_rev,
        'fcrd_up_early_rev':  fcrd_up_early_rev,
        'fcrd_dn_early_rev':  fcrd_dn_early_rev,
        'fcrd_up_late_rev':   fcrd_up_late_rev,
        'fcrd_dn_late_rev':   fcrd_dn_late_rev,
        'buyback_up_cost':    buyback_up_cost,
        'buyback_dn_cost':    buyback_dn_cost,
        'fcrd_up_net':  fcrd_up_early_rev  + fcrd_up_late_rev  - buyback_up_cost,
        'fcrd_dn_net':  fcrd_dn_early_rev  + fcrd_dn_late_rev  - buyback_dn_cost,
        # Hourly vectors for plots
        'early_up_res': A(early_up), 'early_dn_res': A(early_dn),
        'final_up_res': A(fin_up),   'final_dn_res': A(fin_dn),
        'topup_up': A(tu_up),  'topup_dn': A(tu_dn),
        'cancel_up': A(cn_up), 'cancel_dn': A(cn_dn),
        'soc': A(fin_soc), 'b_ch': A(fin_bch), 'b_dis': A(fin_bdis),
        'p_im': A(fin_im), 'p_ex': A(fin_ex),
    }
    r['total_fcrd_rev'] = r['fcrd_up_net'] + r['fcrd_dn_net']

    # ── Base case: battery only (no FCR-D) ────────────────────────────────
    if run_base:
        z24 = np.zeros(T_MAX)
        mb = build_early_model(loads, pvs, buy, sell, z24, z24, af_u, af_d, z24, z24)
        solve_model(mb)
        base_im = xvar2d(mb.p_im, n_ec).sum(axis=0)
        base_ex = xvar2d(mb.p_ex, n_ec).sum(axis=0)
        r['base_energy_cost'] = (buy * base_im - sell * base_ex).sum()

        # No-battery baseline
        net_load = loads.sum(axis=0) - pvs.sum(axis=0)
        r['no_battery_cost'] = (buy * np.maximum(net_load, 0)
                                - sell * np.maximum(-net_load, 0)).sum()

    return r

print('solve_day() defined')

## 8. Single day example: May 11, 2025

In [ ]:
example_date = dt_date(2025, 5, 11)

df_day = df_all[df_all['date'] == example_date].sort_values('hour_utc').reset_index(drop=True)
assert len(df_day) == 24, f'Expected 24 hours, got {len(df_day)} for {example_date}'

loads_day, pvs_day = get_ec_profiles(df_day, ec_types, scale_factors)

res = solve_day(example_date, df_day, loads_day, pvs_day)

print(f'\n=== {example_date} (perfect forecasts) ===')
print(f'No battery cost:        {res["no_battery_cost"]:>10.1f} DKK')
print(f'Base energy cost:       {res["base_energy_cost"]:>10.1f} DKK')
print(f'Net energy cost (FCRD): {res["net_energy_cost"]:>10.1f} DKK')
print(f'FCR-D Up net:           {res["fcrd_up_net"]:>10.1f} DKK')
print(f'FCR-D Down net:         {res["fcrd_dn_net"]:>10.1f} DKK')
print(f'Total FCR-D:            {res["total_fcrd_rev"]:>10.1f} DKK')

In [ ]:
# ── Figure 2: Arbitrage view ─────────────────────────────────────────────
hours = np.arange(24)
buy_p  = df_day[COL_BUY].values
sell_p = df_day[COL_SELL].values

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle(f'Production, Consumption & Prices — {example_date}', fontsize=13)

ax = axes[0]
agg_pv   = pvs_day.sum(axis=0)
agg_load = loads_day.sum(axis=0)
net_batt = res['b_ch'] - res['b_dis']
ax.bar(hours, agg_pv, color='gold', alpha=0.7, label='PV production')
ax.bar(hours, -agg_load, color='steelblue', alpha=0.7, label='Demand (−)')
ax.step(hours, net_batt, color='green', lw=1.5, where='mid', label='Net battery (ch−dis)')
ax.set_ylabel('kW')
ax.legend(loc='upper left', fontsize=8)
ax2 = ax.twinx()
ax2.plot(hours, buy_p, 'r--', lw=1, label='Buy price')
ax2.plot(hours, sell_p, 'b--', lw=1, label='Sell price')
ax2.set_ylabel('Price (DKK/kWh)')
ax2.legend(loc='upper right', fontsize=8)

ax = axes[1]
ax.plot(hours, res['soc'], 'r-o', ms=4, label='SOC')
ax.axhline(S_MAX_PORTFOLIO, color='grey', ls=':', label=f'Cap ({S_MAX_PORTFOLIO} kWh)')
ax.set_ylabel('SOC (kWh)'); ax.set_xlabel('Hour (UTC)')
ax.legend(loc='upper left', fontsize=8)
ax3 = ax.twinx()
ax3.plot(hours, buy_p, 'r--', lw=1); ax3.plot(hours, sell_p, 'b--', lw=1)
ax3.set_ylabel('Price (DKK/kWh)')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'arbitrage.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── 1. COLOR CONFIGURATION & PLT RC PARAMS ─────────────────────────
# Exact hex colors from image 3
COLOR_DEMAND    = '#00BDF2'  # Cyan
COLOR_BATTERY   = '#F42CFF'  # Magenta / SOC
COLOR_PV        = '#FFCC00'  # Gold / Yellow
COLOR_BUY       = '#000000'  # Black
COLOR_SELL      = '#787878'  # Gray
COLOR_FORB_UP   = '#E0E0E0'  # Light gray background band
COLOR_FORB_DN   = '#9E9E9E'  # Darker gray background band

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#CCCCCC',
    'axes.grid':         False,      # Dropped the grid background
    'grid.color':        '#F0F0F0',
    'grid.linewidth':    0.6,
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif', 'Liberation Serif', 'Times New Roman', 'serif'],
    'figure.dpi':        150,
})

# ── 2. DATA PREPARATION (MANUAL INTERPOLATION FROM CHECKS) ─────────
hours = np.arange(24)
hour_labels = [f"{h:02d}" for h in hours]

agg_pv   = pvs_day.sum(axis=0)
agg_load = loads_day.sum(axis=0)
net_batt = res['b_ch'] - res['b_dis']

buy_p  = df_day[COL_BUY].values
sell_p = df_day[COL_SELL].values

# Corridor calculations
ya_u = df_day['y_acc_up'].values
ya_d = df_day['y_acc_down'].values
forb_up = ya_u * T_SUSTAIN / ETA * res['final_up_res']
forb_dn = ya_d * T_SUSTAIN * ETA * res['final_dn_res']

# ── 3. PLOT CREATION ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 11), sharex=True)

# Main Super Title
fig.suptitle(f'Production, Consumption & Prices — {example_date}', 
             fontsize=20, fontweight='bold', fontfamily='serif', y=0.96)

# ── TOP SUBPLOT: POWERS (kW) ───────────────────────────────────────
ax1 = axes[0]
width = 0.35  # Adjusted bar width for cleaner spacing

# Bars and steps
ax1.bar(hours - width/2, agg_pv, width=width, color=COLOR_PV, label='PV production', edgecolor='none')
ax1.bar(hours - width/2, -agg_load, width=width, color=COLOR_DEMAND, label='Demand (−)', edgecolor='none')
ax1.bar(hours + width/2, net_batt, width=width, color=COLOR_BATTERY, label='Net battery (ch−dis)', edgecolor='none')

ax1.set_ylabel('kW', fontsize=16, fontfamily='serif')
ax1.tick_params(axis='y', labelsize=16)
ax1.axhline(0, color='black', linewidth=0.5, zorder=1) # Baseline 0

# Twin axis for prices (Top)
ax1_twin = ax1.twinx()
ax1_twin.plot(hours, buy_p, color=COLOR_BUY, linestyle='--', marker='o', ms=2, lw=1, label='Buy price')
ax1_twin.plot(hours, sell_p, color=COLOR_SELL, linestyle=':', marker='s', ms=2, lw=1, label='Sell price')
ax1_twin.set_ylabel('Price (DKK/kWh)', fontsize=16, fontfamily='serif')
ax1_twin.tick_params(axis='y', labelsize=16)

# Legends for top subplot
handles1, labels1 = ax1.get_legend_handles_labels()
handles1_t, labels1_t = ax1_twin.get_legend_handles_labels()
ax1.legend(handles1, labels1, loc='upper left', fontsize=11, framealpha=0.85)
ax1_twin.legend(handles1_t, labels1_t, loc='upper right', fontsize=11, framealpha=0.85)


# ── BOTTOM SUBPLOT: SOC & CORRIDOR (kWh) ───────────────────────────
ax2 = axes[1]

# Forbidden Bands (Plotted first to stay in the background)
ax2.fill_between(hours, 0, forb_up, color=COLOR_FORB_UP, alpha=0.6, label='Forbidden (Up Sustain)', edgecolor='none')
ax2.fill_between(hours, S_MAX_PORTFOLIO - forb_dn, S_MAX_PORTFOLIO, 
                 color=COLOR_FORB_DN, alpha=0.8, label='Forbidden (Dn Sustain)', edgecolor='none')

# Capacity reference line
ax2.axhline(S_MAX_PORTFOLIO, color='gray', linestyle=':', linewidth=0.8, label=f'Cap ({S_MAX_PORTFOLIO} kWh)')

# State of Charge (SOC) line
ax2.plot(hours, res['soc'], color=COLOR_BATTERY, linestyle='-', marker='o', ms=3, lw=1.5, label='SOC')

ax2.set_ylabel('SOC (kWh)', fontsize=16, fontfamily='serif')
ax2.set_xlabel('Hour (UTC)', fontsize=16, fontfamily='serif')
ax2.tick_params(axis='y', labelsize=16)

# Twin axis for prices (Bottom)
ax2_twin = ax2.twinx()
ax2_twin.plot(hours, buy_p, color=COLOR_BUY, linestyle='--', marker='o', ms=2, lw=1, label='Buy price')
ax2_twin.plot(hours, sell_p, color=COLOR_SELL, linestyle=':', marker='s', ms=2, lw=1, label='Sell price')
ax2_twin.set_ylabel('Price (DKK/kWh)', fontsize=16, fontfamily='serif')
ax2_twin.tick_params(axis='y', labelsize=16)

# Legends for bottom subplot
handles2, labels2 = ax2.get_legend_handles_labels()
handles2_t, labels2_t = ax2_twin.get_legend_handles_labels()
ax2.legend(handles2, labels2, loc='upper left', fontsize=11, framealpha=0.85)


# ── 4. CLEANUP SPINES & TICKS (MATCHING STYLE) ─────────────────────
for ax in [ax1, ax2, ax1_twin, ax2_twin]:
    ax.spines['top'].set_visible(False)
    ax.grid(False) # Ensures grid lines are hidden explicitly on each axis

# Format X-axis labels exactly like the second image ('00', '01', ...)
ax2.set_xticks(hours)
ax2.set_xticklabels(hour_labels, rotation=45, fontsize=11, fontfamily='serif')

# Tighten frame and save
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'arbitrage_polished.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: Sequential auction reservations ────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(f'Sequential Auction — {example_date}', fontsize=13)
w = 0.35

ax = axes[0]; ax.set_title('Early auction reservations (perfect forecasts)')
ax.bar(hours-w/2, res['early_up_res'], w, color='salmon', label='Early Up')
ax.bar(hours+w/2, res['early_dn_res'], w, color='mediumaquamarine', label='Early Down')
ax.axhline(P_MIN_BID, color='grey', ls=':', label=f'Min bid ({P_MIN_BID} kW)')
ax.set_ylabel('kW'); ax.legend(fontsize=8)

ax = axes[1]; ax.set_title('Final reservations after late auction')
ax.bar(hours-w/2, res['final_up_res'], w, color='salmon', label='Final Up')
ax.bar(hours+w/2, res['final_dn_res'], w, color='mediumaquamarine', label='Final Down')
ax.set_ylabel('kW'); ax.legend(fontsize=8)

ax = axes[2]; ax.set_title('Late auction adjustments')
ax.bar(hours-w/2, res['topup_up'], w, color='salmon', alpha=0.7, label='Top-up Up')
ax.bar(hours-w/2, -res['cancel_up'], w, color='darkred', alpha=0.5, label='Cancel Up', hatch='//')
ax.bar(hours+w/2, res['topup_dn'], w, color='mediumaquamarine', alpha=0.7, label='Top-up Down')
ax.bar(hours+w/2, -res['cancel_dn'], w, color='teal', alpha=0.5, label='Cancel Down', hatch='//')
ax.set_ylabel('kW'); ax.set_xlabel('Hour (UTC)'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'reservations1.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 5: SOC & Feasibility Corridor ─────────────────────────────────
ya_u = df_day['y_acc_up'].values
ya_d = df_day['y_acc_down'].values
forb_up = ya_u * T_SUSTAIN / ETA * res['final_up_res']
forb_dn = ya_d * T_SUSTAIN * ETA * res['final_dn_res']

up_e = df_day[COL_UP_EARLY].values
up_l = df_day[COL_UP_LATE].values
dn_e = df_day[COL_DOWN_EARLY].values
dn_l = df_day[COL_DOWN_LATE].values

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

ax = axes[0]; ax.set_title('SOC & Feasibility Corridor')
ax.fill_between(hours, 0, forb_up, alpha=0.3, color='salmon', label='Forbidden (Up Sustain)')
ax.fill_between(hours, S_MAX_PORTFOLIO-forb_dn, S_MAX_PORTFOLIO,
                alpha=0.3, color='mediumaquamarine', label='Forbidden (Dn Sustain)')
ax.plot(hours, res['soc'], 'r-o', ms=5, label='SOC')
ax.axhline(S_MAX_PORTFOLIO, color='grey', ls=':')
ax.set_ylabel('SOC (kWh)'); ax.legend(fontsize=8)

ax = axes[1]; ax.set_title('Late Auction Adjustments + FCR-D Prices')
ax.bar(hours-w/2, res['topup_up'], w, color='salmon', label='Top-up Up')
ax.bar(hours+w/2, res['topup_dn'], w, color='mediumaquamarine', label='Top-up Down')
ax.bar(hours-w/2, -res['cancel_up'], w, color='darkred', alpha=0.6, label='Cancel Up', hatch='//')
ax.bar(hours+w/2, -res['cancel_dn'], w, color='teal', alpha=0.6, label='Cancel Down', hatch='//')
ax.set_ylabel('Adjustment (kW)')
ax.set_xlabel('Hour (UTC)')

ax2 = ax.twinx()
ax2.plot(hours, up_e, 'r--', lw=1.5, label='Up early price')
ax2.plot(hours, up_l, 'r-',  lw=1.5, label='Up late price')
ax2.plot(hours, dn_e, 'b--', lw=1.5, label='Down early price')
ax2.plot(hours, dn_l, 'b-',  lw=1.5, label='Down late price')
ax2.set_ylabel('FCR-D price (DKK/kWh)')

handles, labels = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles + handles2, labels + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dashboard2.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ── 1. COLOR CONFIGURATION & THEME ─────────────────────────────────
COLOR_UP        = '#FFCC00'  # Gold / Yellow
COLOR_DOWN      = '#00BDF2'  # Cyan / Light Blue
COLOR_CANCEL_UP = '#B89200'  # Darker gold for cancel outline/hatch
COLOR_CANCEL_DN = '#0083A8'  # Darker cyan for cancel outline/hatch

# Price line colors
COLOR_PRICE_UP_LATE = '#B38600'
COLOR_PRICE_DN_LATE = '#007A99'

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#CCCCCC',
    'axes.grid':         False,      # Disabled global grid
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif', 'Liberation Serif', 'Times New Roman', 'serif'],
    'figure.dpi':        150,
})

# ── 2. DATA UNIFICATION ────────────────────────────────────────────
hours = np.arange(24)
hour_labels = [f"{h:02d}" for h in hours]
w = 0.35  # Clean side-by-side bar spacing width

# Extract price vectors 
up_e = df_day[COL_UP_EARLY].values
up_l = df_day[COL_UP_LATE].values
dn_e = df_day[COL_DOWN_EARLY].values
dn_l = df_day[COL_DOWN_LATE].values

# ── 3. PLOT CREATION ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharex=True)

# Main Super Title
fig.suptitle(f'Sequential Auction — {example_date}', 
             fontsize=20, fontweight='bold', y=0.97)

# ── PANEL 1: EARLY AUCTION RESERVATIONS ────────────────────────────
ax0 = axes[0]
ax0.set_title('Early auction reservations (perfect forecasts)', fontsize=15, fontweight='bold', pad=10)
ax0.bar(hours - w/2, res['early_up_res'], w, color=COLOR_UP, label='Early Up', edgecolor='none')
ax0.bar(hours + w/2, res['early_dn_res'], w, color=COLOR_DOWN, label='Early Down', edgecolor='none')
ax0.axhline(P_MIN_BID, color='gray', linestyle=':', linewidth=0.8, label=f'Min bid ({P_MIN_BID} kW)')
ax0.set_ylabel('kW', fontsize=13)
ax0.tick_params(axis='y', labelsize=14)
ax0.legend(loc='upper left', fontsize=11, framealpha=0.85)

# ── PANEL 2: FINAL RESERVATIONS ───────────────────────────────────
ax1 = axes[1]
ax1.set_title('Final reservations after late auction', fontsize=15, fontweight='bold', pad=10)
ax1.bar(hours - w/2, res['final_up_res'], w, color=COLOR_UP, label='Final Up', edgecolor='none')
ax1.bar(hours + w/2, res['final_dn_res'], w, color=COLOR_DOWN, label='Final Down', edgecolor='none')
ax1.set_ylabel('kW', fontsize=13)
ax1.tick_params(axis='y', labelsize=14)
ax1.legend(loc='upper left', fontsize=11, framealpha=0.85)

# ── PANEL 3: LATE AUCTION ADJUSTMENTS + PRICES ─────────────────────
ax2 = axes[2]
ax2.set_title('Late Auction Adjustments + FCR-D Prices', fontsize=15, fontweight='bold', pad=10)

# Adjustments (Positive Top-ups and Negative Cancels)
ax2.bar(hours - w/2, res['topup_up'], w, color=COLOR_UP, label='Top-up Up', edgecolor='none')
ax2.bar(hours + w/2, res['topup_dn'], w, color=COLOR_DOWN, label='Top-up Down', edgecolor='none')

ax2.bar(hours - w/2, -res['cancel_up'], w, color='none', edgecolor=COLOR_CANCEL_UP, 
        hatch='///', linewidth=1, label='Cancel Up')
ax2.bar(hours + w/2, -res['cancel_dn'], w, color='none', edgecolor=COLOR_CANCEL_DN, 
        hatch='\\\\\\', linewidth=1, label='Cancel Down')

ax2.set_ylabel('Adjustment (kW)', fontsize=13)
ax2.set_xlabel('Hour (UTC)', fontsize=14)
ax2.tick_params(axis='y', labelsize=14)
ax2.axhline(0, color='gray', linewidth=0.6, zorder=1) # Baseline zero line

# Twin X Axis for Prices on the Bottom Panel
ax2_twin = ax2.twinx()
ax2_twin.plot(hours, up_e, color=COLOR_UP, linestyle='--', lw=1.5, label='Up early price')
ax2_twin.plot(hours, up_l, color=COLOR_PRICE_UP_LATE, linestyle='-',  lw=1.8, label='Up late price')
ax2_twin.plot(hours, dn_e, color=COLOR_DOWN, linestyle='--', lw=1.5, label='Down early price')
ax2_twin.plot(hours, dn_l, color=COLOR_PRICE_DN_LATE, linestyle='-',  lw=1.8, label='Down late price')
ax2_twin.set_ylabel('FCR-D price (DKK/kWh)', fontsize=13)
ax2_twin.tick_params(axis='y', labelsize=14)

# Consolidated Legend for Panel 3
handles, labels = ax2.get_legend_handles_labels()
handles2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(handles + handles2, labels + labels2, loc='upper left', fontsize=10, framealpha=0.85)

# ── 4. CLEANUP SPINES & TICKS ──────────────────────────────────────
for ax in [ax0, ax1, ax2, ax2_twin]:
    ax.spines['top'].set_visible(False)
    ax.grid(False) # Explicitly turning off the grid for each axis

# Clear right spines except for the twin target
ax0.spines['right'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Shared clean X-axis labeling format ('00', '01', ...)
ax2.set_xticks(hours)
ax2.set_xticklabels(hour_labels, rotation=45, fontsize=12)

# Tight layout reserving space at the top for fig.suptitle
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(FIGURES_DIR, 'sequential_auction_combined.png'), dpi=150, bbox_inches='tight')
plt.show()

## Diagnostic: Up vs Down reservation — does C1 ever bid both directions?

For C1, the NEM constraint gives: `p_up + 0.2×p_down ≤ 100 kW` and `p_down + 0.2×p_up ≤ 100 kW`. Bidding 100 kW in both directions requires 120 kW of inverter capacity, but C1 only has 100 kW. So both directions simultaneously should be infeasible — this plot confirms it.

In [ ]:

hours = np.arange(24)
up_res = res['final_up_res']
dn_res = res['final_dn_res']

# Hours where both > 0 (should be none for C1)
both_active = (up_res > 1.0) & (dn_res > 1.0)
only_up     = (up_res > 1.0) & (dn_res <= 1.0)
only_dn     = (dn_res > 1.0) & (up_res <= 1.0)
neither     = (up_res <= 1.0) & (dn_res <= 1.0)

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
fig.suptitle(
    f'C1 (100 kW / 200 kWh) — FCR-D Reservation Diagnostic\n{example_date}  |  '
    f'p_min = {P_MIN_BID} kW  |  Inverter = {P_MAX_PORTFOLIO} kW',
    fontsize=12
)

# ── Panel 1: up & down reservation side-by-side ──────────────────────────
ax = axes[0]
w = 0.38
bars_up = ax.bar(hours - w/2, up_res, w, color='#C44E52', alpha=0.85, label='FCR-D↑ reserved')
bars_dn = ax.bar(hours + w/2, dn_res, w, color='#4C72B0', alpha=0.85, label='FCR-D↓ reserved')
ax.axhline(P_MIN_BID, color='black', ls='--', lw=1.2, label=f'Min bid ({P_MIN_BID} kW)')
ax.axhline(P_MAX_PORTFOLIO, color='grey', ls=':', lw=1, label=f'Inverter cap ({P_MAX_PORTFOLIO} kW)')

# Highlight both-active hours in red (should not exist for C1)
for h in np.where(both_active)[0]:
    ax.axvspan(h - 0.5, h + 0.5, color='red', alpha=0.25, zorder=0)

ax.set_ylabel('Reserved (kW)')
ax.set_title('Final reservation per hour (⚠ red = BOTH directions active)')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, P_MAX_PORTFOLIO * 1.2)

# ── Panel 2: NEM headroom used ───────────────────────────────────────────
ax2 = axes[1]
# Discharge side: up_res + 0.2*dn_res  (must be ≤ 100 kW)
# Charge side:    dn_res + 0.2*up_res  (must be ≤ 100 kW)
dis_used = up_res + UPSILON_P * dn_res
ch_used  = dn_res + UPSILON_P * up_res

ax2.bar(hours - w/2, dis_used, w, color='#C44E52', alpha=0.75, label='Discharge side used\n(up + 0.2×dn)')
ax2.bar(hours + w/2, ch_used,  w, color='#4C72B0', alpha=0.75, label='Charge side used\n(dn + 0.2×up)')
ax2.axhline(P_MAX_PORTFOLIO, color='black', ls='--', lw=1.5,
            label=f'Inverter cap = {P_MAX_PORTFOLIO} kW')
ax2.set_ylabel('Effective kW used')
ax2.set_title('NEM constraint utilisation — must stay ≤ inverter cap')
ax2.legend(fontsize=8, loc='upper right')
ax2.set_ylim(0, P_MAX_PORTFOLIO * 1.3)

# Annotate any hour that exceeds the cap
for h in hours:
    if dis_used[h] > P_MAX_PORTFOLIO + 0.5 or ch_used[h] > P_MAX_PORTFOLIO + 0.5:
        ax2.text(h, max(dis_used[h], ch_used[h]) + 2, '!',
                 ha='center', color='red', fontweight='bold', fontsize=12)

# ── Panel 3: direction chosen per hour with prices ───────────────────────
ax3 = axes[2]
# Colour each hour bar by what was bid
bar_colors = []
for h in hours:
    if both_active[h]:   bar_colors.append('red')
    elif only_up[h]:     bar_colors.append('#C44E52')
    elif only_dn[h]:     bar_colors.append('#4C72B0')
    else:                bar_colors.append('#dddddd')

total_res = up_res + dn_res
ax3.bar(hours, total_res, 0.7, color=bar_colors, alpha=0.85, zorder=2)
ax3.axhline(P_MIN_BID, color='black', ls='--', lw=1.2)
ax3.set_ylabel('Total reserved (kW)')
ax3.set_xlabel('Hour (UTC)')
ax3.set_title(
    f'Direction chosen each hour   '
    f'[Red=↑  Blue=↓  Grey=no bid  Dark red=BOTH (should not exist)]'
)

ax3b = ax3.twinx()
up_e = df_day[COL_UP_EARLY].values
dn_e = df_day[COL_DOWN_EARLY].values
ax3b.plot(hours, up_e, 'r--', lw=1.5, label='FCR-D↑ early price')
ax3b.plot(hours, dn_e, 'b--', lw=1.5, label='FCR-D↓ early price')
ax3b.set_ylabel('Price (DKK/kWh)')
ax3b.legend(fontsize=8, loc='upper left')

ax3.set_xticks(hours)

# Summary annotation
n_up   = only_up.sum()
n_dn   = only_dn.sum()
n_both = both_active.sum()
n_none = neither.sum()
summary = (f'Hours: ↑only={n_up}  ↓only={n_dn}  '
           f'both={n_both} (expected 0)  none={n_none}')
ax3.text(0.01, 0.97, summary, transform=ax3.transAxes,
         fontsize=9, va='top', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'C1_direction_diagnostic.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Print hour-by-hour table
print(f"\n{'Hour':>5}  {'Up res':>8}  {'Dn res':>8}  {'dis_used':>10}  {'ch_used':>10}  Direction")
print('-' * 60)
for h in hours:
    if both_active[h]: d = 'BOTH ⚠'
    elif only_up[h]:   d = 'UP ↑'
    elif only_dn[h]:   d = 'DOWN ↓'
    else:              d = 'none'
    print(f"{h:>5}  {up_res[h]:>8.1f}  {dn_res[h]:>8.1f}  "
          f"{dis_used[h]:>10.1f}  {ch_used[h]:>10.1f}  {d}")
